Leitura do Dataset


In [0]:
df_basics = spark.read.csv(
    "dbfs:/Volumes/workspace/mvp/filmes_imdb/title.basics.tsv.gz",
    sep="\t",
    header=True,
    inferSchema=True
)

df_ratings = spark.read.csv(
    "dbfs:/Volumes/workspace/mvp/filmes_imdb/title.ratings.tsv.gz",
    sep="\t",
    header=True,
    inferSchema=True
)


Transformação de Dados

Filtrar apenas filmes. Selecionar colunas relevantes. 

In [0]:
from pyspark.sql.functions import col

df_filmes_tratado = (
    df_basics
    .filter(col("titleType") == "movie")
    .select(
        "tconst",
        "primaryTitle",
        "startYear",
        "runtimeMinutes",
        "genres"
    )
)


In [0]:
df_basics.printSchema()


root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: integer (nullable = true)
 |-- startYear: string (nullable = true)
 |-- endYear: string (nullable = true)
 |-- runtimeMinutes: string (nullable = true)
 |-- genres: string (nullable = true)



Filtrando Nulos

In [0]:
#Criando Views

df_filmes_tratado.createOrReplaceTempView("filmes")
df_ratings.createOrReplaceTempView("ratings")



In [0]:
%sql
-- Verificando Nulos - Tabela Filmes

SELECT
  COUNT(*) AS total_registros,
  SUM(CASE WHEN startYear IS NULL THEN 1 ELSE 0 END) AS startYear_nulos,
  SUM(CASE WHEN runtimeMinutes IS NULL THEN 1 ELSE 0 END) AS runtime_nulos,
  SUM(CASE WHEN genres IS NULL THEN 1 ELSE 0 END) AS genres_nulos
FROM filmes; 

total_registros,startYear_nulos,runtime_nulos,genres_nulos
734427,0,0,0


In [0]:
%sql
-- Verificando Nulos - Tabela Rating 

SELECT
  COUNT(*) AS total_registros,
  SUM(CASE WHEN averageRating IS NULL THEN 1 ELSE 0 END) AS rating_nulos,
  SUM(CASE WHEN numVotes IS NULL THEN 1 ELSE 0 END) AS votos_nulos
FROM ratings;


total_registros,rating_nulos,votos_nulos
1611810,0,0


In [0]:
%sql
--Verificando a quantidade de filmes

SELECT COUNT(*) AS total_filmes
FROM filmes;


total_filmes
734427


In [0]:
%sql
-- Quais gêneros tem maior nota de avaliação

SELECT
  f.genres,
  AVG(r.averageRating) AS media_avaliacao,
  COUNT(*) AS quantidade_filmes
FROM filmes f
JOIN ratings r
  ON f.tconst = r.tconst
GROUP BY f.genres
ORDER BY media_avaliacao DESC;


genres,media_avaliacao,quantidade_filmes
"Documentary,Fantasy,Thriller",9.7,1
"Comedy,Drama,Talk-Show",9.4,1
"Adult,Reality-TV",9.3,1
"Documentary,News,Reality-TV",9.25,2
"Family,Music,Sport",9.1,1
"Documentary,Reality-TV,Romance",9.0,1
"Adventure,Animation,Sport",8.8,1
"Documentary,Musical,Thriller",8.7,1
"Action,Animation,Game-Show",8.7,1
"Documentary,Music,Sci-Fi",8.7,1


In [0]:
%sql
-- Relação do ano de lançamento com avaliação 

SELECT startYear, AVG(averageRating)
FROM filmes f
JOIN ratings r ON f.tconst = r.tconst
GROUP BY startYear
ORDER BY startYear;


startYear,AVG(averageRating)
1894,5.2
1896,3.1
1897,4.65
1898,2.316666666666667
1899,2.375
1900,3.2375
1901,2.8000000000000003
1902,1.7666666666666666
1903,3.7
1904,3.6


In [0]:
%sql
--Relação do ano de lançamento com avaliação por década

SELECT
  FLOOR(CAST(f.startYear AS INT) / 10) * 10 AS decada,
  ROUND(AVG(r.averageRating), 2) AS media_avaliacao,
  COUNT(*) AS quantidade_filmes
FROM filmes f
JOIN ratings r
  ON f.tconst = r.tconst
WHERE f.startYear IS NOT NULL
  AND f.startYear <> '\\N'
GROUP BY FLOOR(CAST(f.startYear AS INT) / 10) * 10
ORDER BY decada;



decada,media_avaliacao,quantidade_filmes
1890,2.81,18
1900,3.56,162
1910,5.78,1960
1920,6.0,4004
1930,6.1,9319
1940,6.19,9241
1950,6.19,13133
1960,6.1,17726
1970,5.94,23530
1980,6.0,26262


In [0]:
%sql
--Relação da duração do filme com a nota obtida

SELECT
  CASE
    WHEN try_cast(f.runtimeMinutes AS INT) < 60 THEN 'Até 60 min'
    WHEN try_cast(f.runtimeMinutes AS INT) BETWEEN 60 AND 89 THEN '60–89 min'
    WHEN try_cast(f.runtimeMinutes AS INT) BETWEEN 90 AND 119 THEN '90–119 min'
    WHEN try_cast(f.runtimeMinutes AS INT) BETWEEN 120 AND 149 THEN '120–149 min'
    ELSE '150 min ou mais'
  END AS faixa_duracao,
  ROUND(AVG(r.averageRating), 2) AS media_avaliacao,
  COUNT(*) AS quantidade_filmes
FROM filmes f
JOIN ratings r
  ON f.tconst = r.tconst
WHERE try_cast(f.runtimeMinutes AS INT) IS NOT NULL
GROUP BY
  CASE
    WHEN try_cast(f.runtimeMinutes AS INT) < 60 THEN 'Até 60 min'
    WHEN try_cast(f.runtimeMinutes AS INT) BETWEEN 60 AND 89 THEN '60–89 min'
    WHEN try_cast(f.runtimeMinutes AS INT) BETWEEN 90 AND 119 THEN '90–119 min'
    WHEN try_cast(f.runtimeMinutes AS INT) BETWEEN 120 AND 149 THEN '120–149 min'
    ELSE '150 min ou mais'
  END
ORDER BY faixa_duracao;


faixa_duracao,media_avaliacao,quantidade_filmes
120–149 min,6.51,26510
150 min ou mais,6.73,8750
60–89 min,6.04,112276
90–119 min,6.1,138078
Até 60 min,6.68,17943


In [0]:
%sql
--Relação entre quantidades de votos e nota

SELECT
  CASE
    WHEN r.numVotes < 100 THEN 'Até 100 votos'
    WHEN r.numVotes BETWEEN 100 AND 999 THEN '100–999 votos'
    WHEN r.numVotes BETWEEN 1000 AND 9999 THEN '1k–9k votos'
    WHEN r.numVotes BETWEEN 10000 AND 99999 THEN '10k–99k votos'
    ELSE '100k votos ou mais'
  END AS faixa_votos,
  ROUND(AVG(r.averageRating), 2) AS media_avaliacao,
  COUNT(*) AS quantidade_filmes
FROM ratings r
JOIN filmes f
  ON r.tconst = f.tconst
WHERE r.numVotes IS NOT NULL
GROUP BY
  CASE
    WHEN r.numVotes < 100 THEN 'Até 100 votos'
    WHEN r.numVotes BETWEEN 100 AND 999 THEN '100–999 votos'
    WHEN r.numVotes BETWEEN 1000 AND 9999 THEN '1k–9k votos'
    WHEN r.numVotes BETWEEN 10000 AND 99999 THEN '10k–99k votos'
    ELSE '100k votos ou mais'
  END
ORDER BY faixa_votos;


faixa_votos,media_avaliacao,quantidade_filmes
100k votos ou mais,7.0,2654
100–999 votos,5.76,94812
10k–99k votos,6.47,9502
1k–9k votos,6.12,35142
Até 100 votos,6.36,194639


In [0]:
%sql
--Quais gêneros concentram os filmes mais populares (com maior número de votos)?

SELECT
  f.genres,
  SUM(r.numVotes) AS total_votos,
  COUNT(*) AS quantidade_filmes
FROM filmes f
JOIN ratings r
  ON f.tconst = r.tconst
WHERE f.genres IS NOT NULL
GROUP BY f.genres
ORDER BY total_votos DESC
LIMIT 10;


genres,total_votos,quantidade_filmes
"Action,Adventure,Sci-Fi",57978888,377
Drama,46325300,56747
"Comedy,Drama,Romance",39544563,5084
"Adventure,Animation,Comedy",37780645,826
"Action,Crime,Drama",34674969,3551
Comedy,33526157,26752
"Comedy,Drama",32982316,10672
"Action,Adventure,Fantasy",32686860,495
"Drama,Romance",30799696,10162
"Action,Adventure,Comedy",28779800,1265
